In [4]:
# ════════════════════════════════════════════════════════════
# Phase 3 — Step 2C (FINAL): Merge ET₀ + CHIRPS → climate_weekly
# Mae Na Rua Sub-District, Phayao | Period: 2020–2024
# ════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from scipy.stats import zscore

# ════════════════════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════════════════════
ET0_FILE    = r'ET0_weekly_phayao_2020_2024.csv'
CHIRPS_FILE = r'CHIRPS_weekly_phayao_2020_2024.csv'
OUT_FILE    = r'climate_weekly_phayao_2020_2024.csv'

# ════════════════════════════════════════════════════════════
# STEP 1 — โหลดและกรอง partial weeks
# ════════════════════════════════════════════════════════════
print("="*55)
print("STEP 1 — โหลดข้อมูล")
print("="*55)

ET0 = pd.read_csv(ET0_FILE)
P   = pd.read_csv(CHIRPS_FILE)

ET0 = ET0[ET0['n_days'] == 7].reset_index(drop=True)
P   = P  [P  ['n_days'] == 7].reset_index(drop=True)

print(f"ET₀   : {len(ET0)} weeks  {ET0.columns.tolist()}")
print(f"CHIRPS: {len(P)} weeks  {P.columns.tolist()}")

# ════════════════════════════════════════════════════════════
# STEP 2 — ตรวจและ impute weeks ที่ CHIRPS ขาดหาย
# ════════════════════════════════════════════════════════════
print("\n" + "="*55)
print("STEP 2 — ตรวจ + Impute CHIRPS weeks ที่ขาด")
print("="*55)

et0_keys = set(zip(ET0['year'], ET0['week']))
p_keys   = set(zip(P['year'],   P['week']))
missing  = sorted(et0_keys - p_keys)

print(f"ET₀ weeks   : {len(et0_keys)}")
print(f"CHIRPS weeks: {len(p_keys)}")
print(f"ขาด {len(missing)} weeks:")

# week-level median จากข้อมูลที่มี
P_median = (P.groupby('week')[['P_mm_week','P_eff_paddy','P_eff_upland']]
             .median().reset_index())

imputed_rows = []
for y, w in missing:
    med = P_median[P_median['week'] == w]
    if len(med) > 0:
        row = {
            'year'        : y,
            'week'        : w,
            'P_mm_week'   : round(float(med['P_mm_week'].values[0]),   3),
            'P_eff_paddy' : round(float(med['P_eff_paddy'].values[0]), 3),
            'P_eff_upland': round(float(med['P_eff_upland'].values[0]),3),
            'n_days'      : 7,
            'imputed'     : 1,
        }
    else:
        row = {'year':y, 'week':w, 'P_mm_week':0.0,
               'P_eff_paddy':0.0, 'P_eff_upland':0.0,
               'n_days':7, 'imputed':1}
    imputed_rows.append(row)
    print(f"  Imputed {y} W{w:02d} → P = {row['P_mm_week']:.1f} mm (week median)")

P['imputed'] = 0
P_full = (pd.concat([P, pd.DataFrame(imputed_rows)], ignore_index=True)
           .sort_values(['year','week'])
           .reset_index(drop=True))

# ════════════════════════════════════════════════════════════
# STEP 3 — Merge
# ════════════════════════════════════════════════════════════
print("\n" + "="*55)
print("STEP 3 — Merge ET₀ + Rainfall")
print("="*55)

climate = ET0.merge(
    P_full[['year','week','P_mm_week','P_eff_paddy',
            'P_eff_upland','imputed']],
    on=['year','week'],
    how='inner'
).sort_values(['year','week']).reset_index(drop=True)

print(f"หลัง merge: {len(climate)} weeks  "
      f"({climate['year'].min()} W{climate['week'].min():02d} "
      f"→ {climate['year'].max()} W{climate['week'].max():02d})")

# ════════════════════════════════════════════════════════════
# STEP 4 — Derived Variables
# ════════════════════════════════════════════════════════════
print("\n" + "="*55)
print("STEP 4 — Derived Variables")
print("="*55)

# 4.1 Water deficit (mm/week)
climate['deficit_paddy_mm']  = (climate['ET0_mm_week']
                                 - climate['P_eff_paddy']).round(2)
climate['deficit_upland_mm'] = (climate['ET0_mm_week']
                                 - climate['P_eff_upland']).round(2)

# 4.2 Rolling 4-week rainfall
climate['P_4week'] = (climate['P_mm_week']
                      .rolling(4, min_periods=4)
                      .sum())

# 4.3 SPI-4 (z-score ภายใน week เดียวกันข้ามปี)
climate['SPI_4'] = (
    climate.groupby('week')['P_4week']
    .transform(lambda x: zscore(x, ddof=1) if len(x) > 1 else 0.0)
    .fillna(0.0)   # ← rolling warmup weeks → 0 (climatological normal)
    .round(3)
)

# 4.4 Drought flag (SPI-4 < -1.0 = moderately dry)
climate['drought_flag'] = (climate['SPI_4'] < -1.0).astype(int)

# 4.5 Aridity Index รายสัปดาห์ (cap ที่ 10 ป้องกัน div-by-zero)
climate['AI_week'] = (
    climate['ET0_mm_week'] / (climate['P_mm_week'] + 0.001)
).clip(upper=10).round(3)

print("Variables ที่คำนวณ:")
print(climate[['deficit_paddy_mm','deficit_upland_mm',
               'P_4week','SPI_4','drought_flag','AI_week']]
      .describe().round(2).to_string())

# ════════════════════════════════════════════════════════════
# STEP 5 — Sanity Check
# ════════════════════════════════════════════════════════════
print("\n" + "="*55)
print("STEP 5 — Sanity Check")
print("="*55)

checks = {
    'ET₀ < 0'         : (climate['ET0_mm_week'] < 0).sum(),
    'P < 0'           : (climate['P_mm_week'] < 0).sum(),
    'P_eff > P'       : (climate['P_eff_paddy'] > climate['P_mm_week']+0.01).sum(),
    'AI_week > 10'    : (climate['AI_week'] > 10).sum(),
    'SPI_4 NaN'       : climate['SPI_4'].isna().sum(),
    'deficit NaN'     : climate['deficit_paddy_mm'].isna().sum(),
    'imputed rows'    : int(climate['imputed'].sum()),
    'total weeks'     : len(climate),
}
all_pass = True
for k, v in checks.items():
    if k == 'imputed rows':
        flag = '📌'
    elif k == 'total weeks':
        flag = '📊'
    else:
        flag = '✅' if v == 0 else '❌'
        if v != 0 and k not in ('imputed rows','total weeks'):
            all_pass = False
    print(f"  {flag}  {k:<22}: {v}")

print(f"\n{'✅ All checks passed!' if all_pass else '❌ มี issue — ตรวจสอบก่อนเดินหน้า'}")

# Seasonal pattern
climate['season'] = climate['week'].apply(
    lambda w: 'Dry-hot(W10-18)' if 10<=w<=18
    else ('Wet(W19-44)' if 19<=w<=44 else 'Dry-cool'))

print("\nค่าเฉลี่ยตามฤดูกาล:")
print(climate.groupby('season')[
    ['ET0_mm_week','P_mm_week','deficit_paddy_mm','AI_week']
].mean().round(2).to_string())

# Annual balance
print("\nAnnual water balance:")
ann = climate.groupby('year')[['ET0_mm_week','P_mm_week']].sum().round(0)
ann['status'] = (ann['ET0_mm_week'] > ann['P_mm_week']).map(
    {True:'⚠️  drought', False:'✅ sufficient'})
print(ann.to_string())

# Drought summary
n_dr = climate['drought_flag'].sum()
print(f"\nDrought weeks (SPI-4 < −1): {n_dr} / {len(climate)} "
      f"({n_dr/len(climate)*100:.1f}%)")

# ════════════════════════════════════════════════════════════
# STEP 6 — Export
# ════════════════════════════════════════════════════════════
print("\n" + "="*55)
print("STEP 6 — Export")
print("="*55)

out_cols = [
    'year','week',
    'ET0_mm_week','T_mean','RH_pct','VPD_kPa','u2_ms','Rn_MJ',
    'P_mm_week','P_eff_paddy','P_eff_upland',
    'deficit_paddy_mm','deficit_upland_mm',
    'P_4week','SPI_4','drought_flag','AI_week',
    'imputed','n_days'
]
climate[out_cols].to_csv(OUT_FILE, index=False)

print(f"✅ Saved : {OUT_FILE}")
print(f"   Rows  : {len(climate)} | Columns: {len(out_cols)}")
print(f"   Columns: {out_cols}")

# ════════════════════════════════════════════════════════════
# STEP 7 — Sample output
# ════════════════════════════════════════════════════════════
print("\n" + "="*55)
print("STEP 7 — Sample Output")
print("="*55)

view_cols = ['year','week','ET0_mm_week','P_mm_week',
             'P_eff_paddy','deficit_paddy_mm','SPI_4',
             'drought_flag','AI_week','imputed']

print("\nWet season peak (W26–30, 2022):")
s1 = climate[(climate['year']==2022) & (climate['week'].between(26,30))]
print(s1[view_cols].to_string(index=False))

print("\nDry season (W10–14, 2023 — drought year):")
s2 = climate[(climate['year']==2023) & (climate['week'].between(10,14))]
print(s2[view_cols].to_string(index=False))

print("\nImputed rows:")
imp = climate[climate['imputed']==1]
print(imp[view_cols].to_string(index=False))

print("\n📌 NEXT: zone delineation → คำนวณ NIR_A และ GIR_B")

STEP 1 — โหลดข้อมูล
ET₀   : 260 weeks  ['year', 'week', 'ET0_mm_week', 'T_mean', 'RH_pct', 'VPD_kPa', 'u2_ms', 'Rn_MJ', 'n_days']
CHIRPS: 255 weeks  ['year', 'week', 'P_mm_week', 'n_days', 'P_eff_paddy', 'P_eff_upland']

STEP 2 — ตรวจ + Impute CHIRPS weeks ที่ขาด
ET₀ weeks   : 260
CHIRPS weeks: 255
ขาด 5 weeks:
  Imputed 2020 W53 → P = 0.0 mm (week median)
  Imputed 2021 W52 → P = 1.8 mm (week median)
  Imputed 2022 W52 → P = 1.8 mm (week median)
  Imputed 2023 W52 → P = 1.8 mm (week median)
  Imputed 2024 W01 → P = 3.2 mm (week median)

STEP 3 — Merge ET₀ + Rainfall
หลัง merge: 260 weeks  (2020 W01 → 2024 W53)

STEP 4 — Derived Variables
Variables ที่คำนวณ:
       deficit_paddy_mm  deficit_upland_mm  P_4week   SPI_4  drought_flag  AI_week
count            260.00             260.00   257.00  260.00        260.00   260.00
mean               0.00               1.50   117.47   -0.00          0.15     4.09
std               32.22              33.02   109.52    0.87          0.35     4.19
m

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import zscore

ET0_FILE    = r'ET0_weekly_phayao_2020_2024.csv'
CHIRPS_FILE = r'CHIRPS_weekly_phayao_2020_2024.csv'
OUT_FILE    = r'climate_weekly_phayao_2020_2024.csv'

ET0 = pd.read_csv(ET0_FILE)
P   = pd.read_csv(CHIRPS_FILE)

ET0 = ET0[ET0['n_days'] == 7].reset_index(drop=True)
P   = P  [P  ['n_days'] == 7].reset_index(drop=True)

# ════════════════════════════════════════════════════════════
# FIX 1 — ตรวจหา weeks ที่ CHIRPS ขาดหาย
# ════════════════════════════════════════════════════════════
print("="*55)
print("FIX 1 — ตรวจ weeks ที่ขาดหาย")
print("="*55)

et0_keys = set(zip(ET0['year'], ET0['week']))
p_keys   = set(zip(P['year'],   P['week']))
missing  = sorted(et0_keys - p_keys)

print(f"ET₀ weeks : {len(et0_keys)}")
print(f"CHIRPS weeks: {len(p_keys)}")
print(f"CHIRPS ขาด {len(missing)} weeks:")
for y, w in missing:
    print(f"  {y} W{w:02d}")

# ════════════════════════════════════════════════════════════
# FIX 2 — Impute weeks ที่ขาด ด้วย median ของ week เดียวกัน
# ════════════════════════════════════════════════════════════
print("\n" + "="*55)
print("FIX 2 — Impute CHIRPS ที่ขาด")
print("="*55)

# สร้าง week-level median จากข้อมูลที่มี
P_median = (P.groupby('week')[['P_mm_week','P_eff_paddy','P_eff_upland']]
             .median().reset_index())

imputed_rows = []
for y, w in missing:
    med = P_median[P_median['week'] == w]
    if len(med) > 0:
        row = {
            'year'        : y,
            'week'        : w,
            'P_mm_week'   : round(float(med['P_mm_week'].values[0]),   3),
            'P_eff_paddy' : round(float(med['P_eff_paddy'].values[0]), 3),
            'P_eff_upland': round(float(med['P_eff_upland'].values[0]),3),
            'n_days'      : 7,
            'imputed'     : 1,
        }
    else:
        row = {'year':y,'week':w,'P_mm_week':0,'P_eff_paddy':0,
               'P_eff_upland':0,'n_days':7,'imputed':1}
    imputed_rows.append(row)
    print(f"  Imputed {y} W{w:02d} → P={row['P_mm_week']:.1f} mm")

P['imputed'] = 0
P_full = pd.concat([P, pd.DataFrame(imputed_rows)], ignore_index=True)
P_full = P_full.sort_values(['year','week']).reset_index(drop=True)

# ════════════════════════════════════════════════════════════
# MERGE
# ════════════════════════════════════════════════════════════
climate = ET0.merge(
    P_full[['year','week','P_mm_week','P_eff_paddy',
            'P_eff_upland','imputed']],
    on=['year','week'], how='inner'
).sort_values(['year','week']).reset_index(drop=True)

print(f"\nหลัง impute + merge: {len(climate)} weeks")

# ════════════════════════════════════════════════════════════
# DERIVED VARIABLES (แก้ AI_week)
# ════════════════════════════════════════════════════════════
climate['deficit_paddy_mm']  = (climate['ET0_mm_week']
                                 - climate['P_eff_paddy']).round(2)
climate['deficit_upland_mm'] = (climate['ET0_mm_week']
                                 - climate['P_eff_upland']).round(2)

climate['P_4week'] = climate['P_mm_week'].rolling(4, min_periods=4).sum()

climate['SPI_4'] = (
    climate.groupby('week')['P_4week']
    .transform(lambda x: zscore(x, ddof=1) if len(x) > 1 else 0.0)
    .round(3)
)

climate['drought_flag'] = (climate['SPI_4'] < -1.0).astype(int)

# FIX: AI_week — cap ที่ 10 (ค่า > 10 หมายถึง extreme dry แล้ว)
climate['AI_week'] = (
    climate['ET0_mm_week'] / (climate['P_mm_week'] + 0.001)
).clip(upper=10).round(3)

# ════════════════════════════════════════════════════════════
# SANITY CHECK
# ════════════════════════════════════════════════════════════
print("\n" + "="*55)
print("SANITY CHECK (final)")
print("="*55)

checks = {
    'ET₀ < 0'       : (climate['ET0_mm_week'] < 0).sum(),
    'P < 0'         : (climate['P_mm_week'] < 0).sum(),
    'P_eff > P'     : (climate['P_eff_paddy'] > climate['P_mm_week']+0.01).sum(),
    'AI_week > 10'  : (climate['AI_week'] > 10).sum(),
    'SPI_4 NaN'     : climate['SPI_4'].isna().sum(),
    'imputed rows'  : climate['imputed'].sum(),
}
for k, v in checks.items():
    flag = '✅' if v == 0 else '⚠️'
    print(f"  {flag}  {k:<20}: {v}")

print("\nSeasonal means:")
climate['season'] = climate['week'].apply(
    lambda w: 'Dry-hot(10-18)' if 10<=w<=18
    else ('Wet(19-44)' if 19<=w<=44 else 'Dry-cool'))
print(climate.groupby('season')[
    ['ET0_mm_week','P_mm_week','deficit_paddy_mm','AI_week']
].mean().round(2).to_string())

print("\nAnnual balance:")
ann = climate.groupby('year')[['ET0_mm_week','P_mm_week']].sum().round(0)
ann['status'] = (ann['ET0_mm_week'] > ann['P_mm_week']).map(
    {True:'⚠️ drought', False:'✅ sufficient'})
print(ann.to_string())

# ════════════════════════════════════════════════════════════
# EXPORT
# ════════════════════════════════════════════════════════════
out_cols = [
    'year','week',
    'ET0_mm_week','T_mean','RH_pct','VPD_kPa','u2_ms','Rn_MJ',
    'P_mm_week','P_eff_paddy','P_eff_upland',
    'deficit_paddy_mm','deficit_upland_mm',
    'P_4week','SPI_4','drought_flag','AI_week',
    'imputed','n_days'
]
climate[out_cols].to_csv(OUT_FILE, index=False)
print(f"\n✅ Saved: {OUT_FILE}")
print(f"   Rows: {len(climate)} | Columns: {len(out_cols)}")

print("\n📌 NEXT: zone delineation → คำนวณ NIR_A และ GIR_B")

FIX 1 — ตรวจ weeks ที่ขาดหาย
ET₀ weeks : 260
CHIRPS weeks: 255
CHIRPS ขาด 5 weeks:
  2020 W53
  2021 W52
  2022 W52
  2023 W52
  2024 W01

FIX 2 — Impute CHIRPS ที่ขาด
  Imputed 2020 W53 → P=0.0 mm
  Imputed 2021 W52 → P=1.8 mm
  Imputed 2022 W52 → P=1.8 mm
  Imputed 2023 W52 → P=1.8 mm
  Imputed 2024 W01 → P=3.2 mm

หลัง impute + merge: 260 weeks

SANITY CHECK (final)
  ✅  ET₀ < 0             : 0
  ✅  P < 0               : 0
  ✅  P_eff > P           : 0
  ✅  AI_week > 10        : 0
  ⚠️  SPI_4 NaN           : 15
  ⚠️  imputed rows        : 5

Seasonal means:
                ET0_mm_week  P_mm_week  deficit_paddy_mm  AI_week
season                                                           
Dry-cool              21.09       4.15             17.77     6.84
Dry-hot(10-18)        33.94      14.64             22.22     5.98
Wet(19-44)            20.96      50.34            -19.31     1.64

Annual balance:
      ET0_mm_week  P_mm_week        status
year                                      
2